### Suggested Gold tables:


- gold.stock_daily_returns	ticker + date	Returns & moving averages
- gold.stock_volatility_monthly	ticker + month	Risk analysis
- gold.stock_volume_anomalies	ticker + date	Unusual activity alerts
- gold.stock_comparison	date	Cross-ticker dashboard

In [0]:
import dlt
from pyspark.sql.functions import col, when, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType, TimestampType

# ============================================================
# Silver Layer — Cleaned & Validated Stock Prices
# ============================================================

silver_schema = StructType([
    StructField("ticker",       StringType(),    nullable=False),
    StructField("date",         DateType(),      nullable=False),
    StructField("open",         DoubleType(),    nullable=True),
    StructField("high",         DoubleType(),    nullable=True),
    StructField("low",          DoubleType(),    nullable=True),
    StructField("close",        DoubleType(),    nullable=False),
    StructField("volume",       LongType(),      nullable=True),
    StructField("ingested_at",  TimestampType(), nullable=True),
    StructField("processed_at", TimestampType(), nullable=True),
])

@dlt.expect_or_drop("no_null_ticker", "ticker IS NOT NULL")
@dlt.expect_or_drop("no_null_date",   "date IS NOT NULL")
@dlt.expect_or_drop("valid_close",    "close > 0")
@dlt.expect_or_drop("valid_volume",   "volume > 0")
@dlt.table(
    comment="Cleaned and typed stock prices — bad rows dropped to quarantine",
    schema=silver_schema,
    partition_cols=["ticker"],
)
def stocks_silver():
    # Cast and select in one step — schema enforces output types
    return (
        dlt.read_stream("workplace.bronze.stocks_bronze")
        .select(
            col("ticker"),
            col("date").cast("date"),
            col("open").cast("double"),
            col("high").cast("double"),
            col("low").cast("double"),
            col("close").cast("double"),
            col("volume").cast("long"),
            col("ingested_at").cast("timestamp"),
            current_timestamp().alias("processed_at"),
        )
        .dropDuplicates(["ticker", "date"])
    )


@dlt.table(
    comment="Rows that failed Silver quality checks",
    partition_cols=["ticker"]
)
def stocks_silver_quarantine():
    # Cast, flag failure reason, filter only bad rows
    return (
        dlt.read_stream("workplace.bronze.stocks_bronze")
        .select(
            col("ticker"),
            col("date").cast("date"),
            col("open").cast("double"),
            col("high").cast("double"),
            col("low").cast("double"),
            col("close").cast("double"),
            col("volume").cast("long"),
            col("ingested_at").cast("timestamp"),
        )
        .withColumn("quarantine_reason",
            when(col("ticker").isNull(), lit("null ticker"))
            .when(col("date").isNull(),  lit("null date"))
            .when(col("close") <= 0,     lit("invalid close"))
            .when(col("volume") <= 0,    lit("invalid volume"))
            .otherwise(lit("unknown"))
        )
        .filter(
            col("ticker").isNull() |
            col("date").isNull()   |
            (col("close") <= 0)    |
            (col("volume") <= 0)
        )
        .withColumn("quarantined_at", current_timestamp())
    )